# 解答例（教員用）　第9回
***
> このノートブックは **模範解答の一例**です．コードの空欄（`___`）を埋め，探索は複数ラウンドに分けて実行し，解答欄には実際に得られた出力の数値を根拠に書いています．
> 乱数を使う箇所は固定していますが，ライブラリのバージョンや実行環境によって数値が **小数第2〜3位ほどずれる**ことがあります．探索型の問題なので，**別の探索経路・別の最良設定でも，根拠が筋の通っていれば正解**です．

# 第9回　正則化
***
> **前提**: 第3回・第7回の回帰分析を発展させ，正則化とロジスティック回帰を学びます。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（機械学習）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
> | **【探索】** | 最適設定を自分で探す | **仮説 → 試行 → 記録 → 次の仮説** を何度も繰り返す（**15回以上**）。全試行が自動で履歴に残る |
> | **【指示書】** | 知識を LLM に渡す | 実験で得た知識を、LLM が使える「指示書」に書き起こす |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. データの読み込みと過学習の観察
2. Ridge / Lasso の最適設定を探索する【探索】
3. ロジスティック回帰の正則化【説明】
4. ロジスティック回帰の最適設定を探索する【探索】
5. 実験で得た知識を LLM への指示書にする【指示書】

---

## 🧪 この回の進め方：最適な設定は「自分で」探す

この回では **「正解の alpha や C」は教えません**。答えは、あなたが何度も試して見つけます。

```
  ①仮説を立てる → ②試す（1回1行の記録が自動で残る） → ③結果を見て仮説を直す → ①へ戻る
   「alphaを上げると    lab.log が記録            「外れた。なぜ？」
    テストが上がるはず」
```

### 試行錯誤の記録帳 `Lab`（最初のコードセルで用意済み）

| 使い方 | 何をするか |
|---|---|
| `lab.df()` | これまでの全試行を表で見る（試行番号・設定・スコア・仮説） |
| `lab.plot(x="alpha", group="model")` | 設定とスコアの関係、ベスト更新の推移をグラフにする |
| `lab.report()` | 探索の自己診断（回数・範囲・端に張り付いていないか・仮説を書いたか） |
| `lab.final_test(...)` | **テストデータでの最終評価。1回しか実行できない** |
| `lab.save()` | 全履歴を CSV に保存する |

### ルール（ここが「AIの理解」の中心）

1. **探索に使ってよいのは検証スコア（交差検証 CV）だけ**。テストデータは最後に **1回だけ**（見てから設定を変えたらデータリーク）。
2. **試行のたびに「仮説」を書く**（何を確かめたい試行か）。やみくもな総当たりは評価されません。
3. **粗く → 細かく**：まず桁を広く振って良さそうな範囲を掴み、次にその周辺を細かく調べる。
4. **ベストが探索範囲の端に来たら、範囲を広げて確かめる**。
5. 上位の設定同士の差が、CV のばらつき（`cv_std`）より小さいなら「差がある」とは言えない。

### 評価の観点

| 観点 | 見られること |
|---|---|
| 試行の量と質 | 15回以上・仮説が毎回ある・粗→細の流れがある |
| 結果の読み取り | 外れた仮説から何を学んだか（試行番号 `#n` を引用して書く） |
| 検証の作法 | テストは1回だけ／CV と テストの差を正しく解釈している |
| 知識の言語化 | 自分の実験結果を、LLM が実行できる指示書に落とせている（問題5） |

---

## この回で学ぶこと

### 過学習（Overfitting）とは何か

モデルが訓練データに**過度に適合**してしまい，未知のデータ（テストデータ）では精度が落ちる現象を「過学習」という。特に特徴量が多い場合（変数の数 > サンプル数），線形モデルでも簡単に過学習が起きる。

```
【過学習の兆候】
訓練 R² = 0.98  テスト R² = 0.42  ← 大きく乖離 → 過学習

【理想的な状態】
訓練 R² = 0.82  テスト R² = 0.79  ← ほぼ同じ → 汎化できている
```

### 正則化の直感的な理解

正則化とは「モデルの係数が大きくなりすぎることにペナルティを課す」仕組みだ。

通常の線形回帰の損失関数：**誤差の二乗和（RSS）**を最小化

正則化では損失関数に「係数の大きさ」を加える：
- **Ridge（L2正則化）**: RSS + α × Σ(係数²)
- **Lasso（L1正則化）**: RSS + α × Σ|係数|

αが大きいほど正則化が強く，係数は0に近づく（モデルが単純になる）。

### Ridge vs Lasso の本質的な違い

| 比較軸 | Ridge（L2） | Lasso（L1） |
|---|---|---|
| 係数の扱い | すべての係数を小さくする（0にはしない） | 不要な特徴量の係数を**完全に0にする** |
| 特徴量選択 | できない | **できる**（スパースモデル） |
| 向いている場面 | 全特徴量が多少は予測に寄与する場合 | 重要な特徴量が少数に絞られる場合 |
| 計算 | 解析解あり（安定） | 反復計算が必要 |

> **卒業研究での活用**: 説明変数が多い（遺伝子発現データ，アンケート，センサーデータなど）場合，Lasso で重要変数を自動選択する手法は非常によく使われる。

### ロジスティック回帰の正則化パラメータ C

scikit-learn の `LogisticRegression` では正則化強度を **C** で指定する。Cは `alpha` の逆数だ：
- **C が小さい** → 正則化が強い → 係数が小さい → シンプルなモデル
- **C が大きい** → 正則化が弱い → 係数が自由 → 複雑なモデル（過学習しやすい）

混乱しやすいので注意：Ridge の `alpha` と `LogisticRegression` の `C` は**逆の関係**だ。

---

## 📚 将来の自分のためのメモ — 大学研究での応用


### どんな研究で使われるか

| 分野 | 具体例 |
|---|---|
| 生物学・医学 | **遺伝子発現データ**（変数が数万・サンプルが数十）→ Lasso で疾患関連遺伝子を選択 |
| 心理学・行動科学 | 多数のアンケート項目から，うつ傾向や学習成果に効く因子を絞り込む |
| 経済学・マーケティング | 価格弾力性，購買予測で変数が多い回帰（Ridge で安定化） |
| 医学診断 | ロジスティック回帰 + L1/L2 正則化でリスク因子のスクリーニング |

### 応用できる場面

- **説明変数がサンプル数より多い**（または多項式展開で膨らんだ）ときの過学習防止
- **解釈可能性**が重要（「どの変数が効いているか」を係数で読みたい）
- 線形関係が**おおまかに**成り立つ連続値・二値予測
- Lasso で**変数選択**し，その後に生物学的検証や追加実験を計画する流れ

### 応用しにくい・向かない場面

- **強い非線形・複雑な相互作用**（画像，音声，自然言語）→ 深層学習や木系の方が適する
- **因果効果の推定**が目的のとき（正則化回帰は「予測」向き。因果は傾向スコア，DAG など別手法）
- 係数の符号だけで「A が B の原因」と結論づけること（相関≠因果）
- サンプルが極端に少なく，かつノイズが大きい場合（正則化だけでは不十分なことも多い）

### 研究で報告するときのポイント

1. **正則化強度（α や C）の選び方**を交差検証で行ったことを書く（第13回と連動）
2. Lasso を使った場合は**選択された変数のリスト**と生物学的/理論的な解釈をセットで示す
3. 訓練 R² とテスト R² の**乖離**を報告し，過学習が抑えられていることを示す

### 関連する発展トピック（調べてみると良い）

- Elastic Net（Ridge + Lasso の中間）
- Group Lasso（遺伝子セット単位の選択）
- ベイズ回帰（不確かさ付きの係数推定）

In [1]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# LogisticRegression の penalty 引数は scikit-learn 1.8 以降で非推奨警告が出る（動作は同じ）ため、警告のみ非表示にする
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_BASE = "https://raw.githubusercontent.com/ShotaYmzk/AI-kadai/main/data"

def load_student_csv(filename: str) -> pd.DataFrame:
    return pd.read_csv(f"{DATA_BASE}/student/{filename}", sep=";")


# ==== 試行錯誤の記録帳（この回の全実験がここに溜まる。中身を読んで使い方を理解しておくこと）====
class Lab:
    """試行錯誤の記録帳。何回でも log() でき、表・グラフ・診断レポートにできる。
    ・スコア（score）は「検証用スコア（CV）」だけを記録する。テストデータは final_test() で1回だけ。"""

    def __init__(self, name, score_name="cv_score", higher_is_better=True):
        self.name, self.score_name, self.hib = name, score_name, higher_is_better
        self.rows, self.test_result = [], None

    def seen(self, params):
        return any(r["params"] == params for r in self.rows)

    def log(self, hypothesis, params, score, **extra):
        """1試行を記録する。同じ params を再度記録しようとすると無視する（重複の水増し防止）。"""
        if self.seen(params):
            print(f"  (skip) 既に試した設定です: {params}")
            return
        self.rows.append(dict(trial=len(self.rows) + 1, hypothesis=hypothesis.strip(),
                              params=dict(params), score=float(score), extra=extra))
        best = self.best()["trial"] == len(self.rows)
        print(f"  #{len(self.rows):02d} {params}  {self.score_name}={score:.4f}"
              + "".join(f"  {k}={v}" for k, v in extra.items()) + ("  ★ベスト更新" if best else ""))

    def df(self):
        import pandas as pd
        return pd.DataFrame([{"trial": r["trial"], **r["params"], self.score_name: round(r["score"], 4),
                              **r["extra"], "hypothesis": r["hypothesis"]} for r in self.rows])

    def best(self):
        return (max if self.hib else min)(self.rows, key=lambda r: r["score"])

    def plot(self, x, group=None, logx=True, query=None):
        """パラメータ x とスコアの関係（group ごとに線を分ける）と、試行回数ごとのベスト更新の推移。"""
        import matplotlib.pyplot as plt
        d = self.df()
        dd = d.query(query) if query else d
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        for g, sub in (dd.groupby(group) if group else [("all", dd)]):
            sub = sub.sort_values(x)
            ax[0].plot(sub[x], sub[self.score_name], "o-", label=str(g))
        if logx and dd[x].dtype.kind in "if" and (dd[x] > 0).all(): ax[0].set_xscale("log")
        ax[0].set_xlabel(x); ax[0].set_ylabel(self.score_name); ax[0].legend(); ax[0].set_title("Parameter vs. validation score")
        cur = d[self.score_name].cummax() if self.hib else d[self.score_name].cummin()
        ax[1].step(d["trial"], cur, where="post"); ax[1].plot(d["trial"], d[self.score_name], "x", alpha=.5)
        ax[1].set_xlabel("Trial number"); ax[1].set_title("Best score so far (flat = plateau)")
        plt.tight_layout(); plt.show()

    def report(self, min_trials=15):
        """探索の質の自己診断：回数・カバー範囲・端に張り付いていないか・仮説を書いたか。"""
        import numpy as np
        d, n = self.df(), len(self.rows)
        print(f"■ 試行回数: {n}（目安 {min_trials} 回以上） → {'OK' if n >= min_trials else 'まだ足りない'}")
        keys = list(self.rows[0]["params"]) if n else []
        b = self.best()
        print(f"■ ベスト: trial#{b['trial']} {b['params']}  {self.score_name}={b['score']:.4f}")
        for k in keys:
            vals = [r["params"][k] for r in self.rows]
            uniq = sorted(set(map(str, vals)))
            line = f"■ {k}: {len(uniq)} 種類を試した"
            nums = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
            if len(nums) == len(vals) and len(set(nums)) > 2:
                edge = k != "seed" and b["params"][k] in (min(nums), max(nums))
                line += f"（範囲 {min(nums):g}〜{max(nums):g}）" + (" ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう" if edge else "")
            print(line)
        w = sum(1 for r in self.rows if len(r["hypothesis"]) >= 8)
        print(f"■ 仮説を書いた試行: {w}/{n}" + ("" if w == n else " ⚠ 何を確かめる試行か毎回書こう"))
        if n >= 6:
            third = n // 3
            print(f"■ 前半の最高 {d[self.score_name][:third].max():.4f} → 後半の最高 {d[self.score_name][-third:].max():.4f}")

    def final_test(self, score_fn):
        """テストデータでの最終評価。1回しか実行できない（2回目以降はエラー）。"""
        if self.test_result is not None:
            raise RuntimeError(f"テストは既に使用済みです（結果 {self.test_result:.4f}）。テストを見て設定を変えるのはデータリークです。")
        self.test_result = float(score_fn())
        print(f"最終テストスコア = {self.test_result:.4f}   （検証ベスト = {self.best()['score']:.4f}）")
        return self.test_result

    def save(self, path=None):
        path = path or f"{self.name}_log.csv"
        self.df().to_csv(path, index=False, encoding="utf-8-sig"); print("保存:", path)

## 問題1　ベースラインの線形回帰と過学習の観察　【骨格+選択】
***

### データについて

UCI の学生成績データには，ポルトガルの数学と国語の成績が含まれている。
- `G1`: 1学期の成績（0〜20点）
- `G2`: 2学期の成績（0〜20点）
- `G3`: 最終成績（0〜20点）← 予測したい目的変数

今回は `G3` を目的変数とし、`G1`・`G2` をはじめとする**多数の数値特徴量**を説明変数に使う。さらに `PolynomialFeatures` で特徴量を多項式展開して数を増やし、「特徴量が多いほど過学習しやすい」状況をわざと作って、正則化の効果を観察する。

### なぜ標準化が必要か（回帰の場合）

線形回帰自体はスケールに影響されないが，正則化（Ridge/Lasso）は**係数の大きさにペナルティ**をかけるため，スケールが異なると不公平なペナルティになる。そこで「多項式展開 → 標準化 → モデル」を **Pipeline** でひとまとめにする（CV の各分割で標準化を正しくやり直すためにも Pipeline が必要）。

### 評価指標の意味

- **R²（決定係数）**: 1 が完璧な予測。「平均値を出すだけの予測」より悪いと**負の値**になる（過学習が激しいと実際に負になる）
- **CV R²**: 訓練データを5分割し，「4/5で学習 → 残り1/5で採点」を5回繰り返した平均。**テストデータを使わずに汎化性能を見積もる**ための値。この回では探索の物差しはすべて CV R² を使う
- **`cv_std`**: 5回の R² のばらつき。**この値より小さい差は「差がある」とは言えない**

### 課題

まず**テストデータを取り分けて金庫にしまいます**（`X_test`, `y_test`。問題2の最後まで触らない）。下のコードは準備が済んでいますが、**核心（CV による評価）はあなたが書きます**（`# ★あなたが書く★`）。

`settings` の `degree` を `1 → 2 → 3` と変えて実行し、**訓練 R²（`train_r2`）と CV R²（`cv_score`）の差**（＝過学習）と **`max_abs_coef`（係数の絶対値の最大）** がどう変わるかを観察してください。**`degree=1` の結果は、正則化なしでどこまで出来るかの基準（ベースライン）**なので必ず残してください（問題2の考察で使います）。試行は自動で `lab` に記録されます。

観察したうえで、次の **設計判断** に答えてください。

> **設計判断1**: 訓練 R² は高いのに CV R² が低い（＝過学習している）状況です。この問題への対処として、次のどれが適切か**1つ選び、理由**を解答用コードセルに書いてください。またそれ以外の選択肢がどうして不適切なのかも解答してください。ここでの選択は問題2の実験につながります。
>
> - **(A) 何もしない（正則化なしの `LinearRegression` のまま）**
> - **(B) Ridge（L2正則化）を使う** … すべての係数を小さく抑える
> - **(C) Lasso（L1正則化）を使う** … 不要な係数を0にして特徴量を絞る
> - **(D) ElasticNet（L1+L2 の両方）を使う** … Ridge と Lasso の中間
> - **(E) 特徴量を減らす（多項式の次数 `degree` を下げる）** … モデル自体を単純にする
> - **(F) 学習データを増やす** … サンプル数を増やして暗記を防ぐ
>
> ヒント：過学習は「係数が極端に大きくなり、訓練データの細部まで暗記してしまう」ことで起きます。`max_abs_coef` と、L2/L1 が係数に何をするか（この回で学ぶこと参照）を根拠にしてください。「正解は1つ」ではなく、**根拠が筋の通った選択**であることが重要です。

### まずデータの中身を見てみよう（実行するだけ）

この回で使う学生成績データ（`student-mat.csv`）は，1行が1人の生徒です．コードを変更せずに実行して，**どんな列があり，どんな値が入っているか**を確認してください．`G1`・`G2`・`G3` は 1学期・2学期・最終の成績（0〜20点），`studytime` は週の勉強時間（1〜4），`failures` は過去の落第回数，`absences` は欠席数です．

In [2]:
# === データの中身を確認する（実行するだけ。コードは変更しない）===
_df = load_student_csv("student-mat.csv")
print("行数・列数:", _df.shape)
print("\n全ての列名:", list(_df.columns))
print("\n先頭 5 行（この回で使う数値の列）:")
print(_df[["G1", "G2", "G3", "studytime", "failures", "absences", "age", "Medu", "Fedu", "goout", "Dalc", "Walc", "health"]].head())
print("\n最終成績 G3 の要約（平均・最小・最大など）:")
print(_df["G3"].describe().round(2))

行数・列数: (395, 33)

全ての列名: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']

先頭 5 行（この回で使う数値の列）:
   G1  G2  G3  studytime  failures  ...  Fedu  goout  Dalc  Walc  health
0   5   6   6          2         0  ...     4      4     1     1       3
1   5   5   6          2         0  ...     1      3     1     1       3
2   7   8  10          2         3  ...     1      2     2     3       3
3  15  14  15          3         0  ...     2      2     1     1       5
4   6  10  10          2         0  ...     3      2     1     2       5

[5 rows x 13 columns]

最終成績 G3 の要約（平均・最小・最大など）:
count    395.00
mean      10.42
std        4.58
min        0.00
25%        8.00
50%       11.00
75%       14.00
max       20.00
Name: G3, d

In [3]:
# === 完成済みコード（★の箇所だけ自分で書く）：試す → lab に記録、を繰り返せる形になっている ===
df = load_student_csv("student-mat.csv")

# 数値特徴量を多めに使う（特徴量が多いほど過学習しやすい）
feature_cols = ["G1", "G2", "studytime", "failures", "absences",
                "age", "Medu", "Fedu", "traveltime", "freetime",
                "goout", "Dalc", "Walc", "health"]
X = df[feature_cols].values
y = df["G3"].values

# テストデータは金庫へ。問題2の「最終評価」まで一切使わない
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
cv5 = KFold(5, shuffle=True, random_state=0)      # 訓練データ内の5分割CV
lab = Lab("q9_regression", score_name="cv_score")  # 問題1・2 の全試行がこの1つの履歴に溜まる

def make_model(name, alpha, degree):
    """多項式展開 → 標準化 → 回帰モデル をつなぐ Pipeline を返す"""
    reg = {"Linear": LinearRegression(),
           "Ridge": Ridge(alpha=alpha),
           "Lasso": Lasso(alpha=alpha, max_iter=10000),
           "ElasticNet": ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=10000)}[name]
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), reg)

def run_trial(hypothesis, name, alpha, degree):
    """1つの設定を試して lab に記録する（同じ設定は2回記録されない）"""
    params = {"model": name, "alpha": alpha, "degree": degree}
    if lab.seen(params):
        print(f"  (skip) 既に試した設定です: {params}"); return
    model = make_model(name, alpha, degree)
    # ★あなたが書く★：訓練データ内の5分割CVで R² を5つ求める（1行）。テストデータは使わない
    #   ヒント: cross_val_score(model, 説明変数, 目的変数, cv=cv5, scoring="r2")
    scores = cross_val_score(model, X_train, y_train, cv=cv5, scoring="r2")
    model.fit(X_train, y_train)
    reg = model[-1]                                    # Pipeline の最後 = 回帰モデル本体
    lab.log(hypothesis, params, scores.mean(),
            cv_std=round(scores.std(), 3),
            train_r2=round(model.score(X_train, y_train), 3),
            n_nonzero=f"{int(np.sum(np.abs(reg.coef_) > 1e-8))}/{len(reg.coef_)}",
            max_abs_coef=round(float(np.abs(reg.coef_).max()), 1))

# === ★ここを変えて実験する★：仮説を書き、settings を書き換えて何度でも実行 ===
# 【変更のしかた】
#  ① hypothesis … 今回の試行で「何を確かめたいか」を1文で書き換える（例：「alpha を10倍にすると CV は上がるはず」）。空・短すぎると report で警告される
#  ② settings   … 試したい設定を「1行に1つのタプル」で並べる。タプルの並び順は (モデル, alpha, degree)
#       モデル … "Linear"（alpha は使わないので 0 と書く）／"Ridge"／"Lasso"／"ElasticNet"
#       alpha  … 正則化の強さ。桁で振る（0.001, 0.01, …, 1000）
#       degree … 多項式の次数 1 / 2 / 3
#     行を足す／消す／数値を書き換えてよい。例：settings = [("Linear", 0, 1), ("Linear", 0, 2), ("Linear", 0, 3)]
#  ③ 実行すると、この settings の各行が1試行として lab に記録される（前に試した試行は消えず、同じ設定は重複記録されない）
#  ④ 出力された各試行の train_r2・cv_score・max_abs_coef を見て、必要なら①②を書き換えて再実行する
#  ※ 問題1 では、degree だけを変えたタプルを 3 つ以上並べる。
hypothesis = "degree を上げるほど訓練R²は上がりCV R²は下がる（過学習）はず"
settings = [("Linear", 0, 1), ("Linear", 0, 2), ("Linear", 0, 3)]   # (モデル, alpha, degree)
for name, alpha, degree in settings:
    run_trial(hypothesis, name, alpha, degree)
q1_n = len(lab.rows)        # 問題1 が終わった時点の試行数（解答セルで使う）

  #01 {'model': 'Linear', 'alpha': 0, 'degree': 1}  cv_score=0.8227  cv_std=0.053  train_r2=0.842  n_nonzero=14/14  max_abs_coef=3.6  ★ベスト更新
  #02 {'model': 'Linear', 'alpha': 0, 'degree': 2}  cv_score=0.6443  cv_std=0.106  train_r2=0.92  n_nonzero=119/119  max_abs_coef=9.6
  #03 {'model': 'Linear', 'alpha': 0, 'degree': 3}  cv_score=-1.1404  cv_std=0.773  train_r2=1.0  n_nonzero=679/679  max_abs_coef=10.3


note you can use here

In [4]:
#note
#

In [5]:
# === ✍️ 問題1 解答（採点対象）===
import pandas as pd

# (1-a) 観察表（自動出力：問題1の試行。degree を 3通り以上試すこと）
observation_1_a = lab.df().iloc[:q1_n]
print(observation_1_a.drop(columns="hypothesis").to_string())

# (1-b) 観察：degree を上げると train_r2 と cv_score の差はどうなったか（数値を引用）
observation_1_b = """
degree=1: train_r2=0.842, cv_score=0.823（差 0.02）。degree=2: train 0.920 / cv 0.644（差 0.28）。degree=3: train 1.000 / cv -1.140（差 2.1 以上）。
degree を上げるほど訓練 R² は 0.84→0.92→1.00 と上がるのに CV R² は 0.82→0.64→-1.14 と下がり、差（過学習）が急拡大した。degree=3 は特徴量 679 個に対し訓練サンプル 316 件で、訓練データを完全に暗記している（train R²=1.0）。CV R² が負＝「平均を出すより悪い」。
"""

# (1-c) 観察：max_abs_coef（係数の絶対値の最大）はどうなったか
observation_1_c = """
max_abs_coef は degree=1: 3.6 → degree=2: 9.6 → degree=3: 10.3 と大きくなった。特徴量が増えて相関の強い列が生まれ、係数が互いに打ち消し合う形で大きな値を取るようになった（過学習の典型的な兆候）。
"""

# (1-d) 設計判断1：過学習への対処として選んだ選択肢（A〜F）：(　)
# (A) 何もしない（正則化なしの LinearRegression のまま）
# (B) Ridge（L2正則化）を使う
# (C) Lasso（L1正則化）を使う
# (D) ElasticNet（L1+L2 の両方）を使う
# (E) 特徴量を減らす（多項式の次数 degree を下げる）
# (F) 学習データを増やす
design1_choice = "B"
# (1-e) その理由（係数の大きさ・L2/L1 が係数に何をするかに触れて）
answer_1_e = """
(B) Ridge を選ぶ。degree=2 で train 0.92 / CV 0.64 と過学習しており max_abs_coef も 9.6 と大きい。過学習は「係数が極端に大きくなって訓練データの細部を暗記する」ことで起きるので、L2 正則化で係数全体を小さく抑えるのが素直な対処。
特徴量は 119 個あるが、G1・G2 以外の多くは弱い相関の変数の組合せなので、「全部が少しずつ効く」と考えて Ridge を第一候補にした（Lasso は不要な特徴量を 0 にするので (C) も有力）。(E) degree を下げるのも有効だが、それは「特徴量を捨てる」ことなので、まずは正則化で試したい。(F) データを増やすのは現実的でない。
"""

   trial   model  alpha  degree  cv_score  cv_std  train_r2 n_nonzero  max_abs_coef
0      1  Linear      0       1    0.8227   0.053     0.842     14/14           3.6
1      2  Linear      0       2    0.6443   0.106     0.920   119/119           9.6
2      3  Linear      0       3   -1.1404   0.773     1.000   679/679          10.3


## 問題2　Ridge / Lasso の最適設定を自分で探す　【探索】
***

### 課題

**「検証スコア（CV R²）が最も高い設定」を、自分で探し当ててください。** 探すものは次の3軸です。

| 軸 | 選択肢 | 注意点 |
|---|---|---|
| `model` | `"Ridge"` / `"Lasso"` / `"ElasticNet"`（比較用に `"Linear"`） | 性質が違う（ヒント：この回で学ぶこと） |
| `alpha` | 正の実数。**桁で効き方が変わる** | 小さすぎ・大きすぎの両方に罠がある |
| `degree` | 1 / 2 / 3 | 増やすと特徴量が爆発的に増える |

`alpha` は正則化の強さ（大きいほど係数は0に近づく）。Lasso は alpha が大きいと**係数が全部0**になり、CV R² が 0 付近に張り付く。Ridge は端が最良になり得る。**やってみないと分かりません。**

### 進め方

1. **【実験前に先に書く】** まず下の「実験前の予測」セルに、最適な model・alpha・degree の予測と理由を書く（LLM に聞いてもよい。その場合は LLM の回答をそのまま貼り、自分の予測との違いも書く）。**実験後に書き直さない。** 予測と結果の差が、あなたが「知識を得た」証拠になる。
2. 次のセルの `hypothesis` と `settings` を書き換えて実行する。**何度でも繰り返す（15回以上）**。記録は `lab` に溜まる。
3. 経過確認セル（`lab.plot` / `lab.report`）で、探索の偏り・端に張り付いていないか・頭打ちかを確認して次の一手を決める。
4. 満足したら、最終評価セルを **1回だけ** 実行する。
5. 解答セルに、探索の進め方・結果の読み取り・考察を書く。

> **考察1**: alpha を大きくすると、CV R² はどう動きましたか？ 小さすぎ・大きすぎのどちらでも下がる「ちょうど良い alpha」がありましたか？
>
> **考察1'（探索範囲）**: ベストが探索範囲の**端**に来たとき、それは何を意味しますか？ Lasso で alpha を大きくしすぎると何が起きますか？（範囲を広げて確かめること）
>
> **考察2（L1 と L2 の違い）**: Ridge の非ゼロ係数の数はほとんど変わらないのに、**Lasso は alpha を大きくすると係数がゼロになっていく**のはなぜですか？「Lasso が特徴量選択になる」とはどういう意味か。（ヒント：L1 は `Σ|係数|`、L2 は `Σ(係数²)` にペナルティをかける。0付近での効き方の違いを考える）
>
> **考察2'（正則化は本当に必要だったか）**: 問題1の `degree=1` の `Linear` の CV R² と、あなたが見つけた最良設定を比べてください。差は `cv_std` と比べて意味のある差ですか？ 設計判断1で選んだ対処（A〜F）は、結果的に最良でしたか？ 手軽で同等の別案はありませんでしたか？

In [6]:
# === ✍️ 問題2-0 実験前の予測（実験を始める前に書く。実験後に書き直さない）===

# 予測するモデル・alpha・degree（それぞれ1つ）
prediction_2_model = "Lasso"
prediction_2_alpha = 1.0
prediction_2_degree = 2
# そう予測した理由（知識・直感・LLMの回答など。LLMに聞いた場合は回答を貼り、自分との違いも書く）
prediction_2_reason = """
degree=2 では特徴量が 119 個あり、G1・G2 以外は弱い変数ばかりなので、不要な特徴量を 0 にする Lasso が有利と予想した。alpha は標準化済みデータで 1 くらいが「ちょうど良い」強さだろうと考えた（LLM に聞いても Lasso alpha=0.1〜1、degree=2 と言われたので、その中間を採用）。
"""

In [7]:
# === ★ここを変えて実験する★：仮説を書き、settings を書き換えて何度でも実行 ===
# 【変更のしかた】
#  ① hypothesis … 今回の試行で「何を確かめたいか」を1文で書き換える（例：「alpha を10倍にすると CV は上がるはず」）。空・短すぎると report で警告される
#  ② settings   … 試したい設定を「1行に1つのタプル」で並べる。タプルの並び順は (モデル, alpha, degree)
#       モデル … "Ridge"／"Lasso"／"ElasticNet"（比較用に "Linear" も可。その場合 alpha は 0）
#       alpha  … 正則化の強さ。モデルごとに良い範囲が違うので、まず桁で広く → 良い桁の周辺を細かく
#       degree … 1 / 2 / 3
#     行を足す／消す／数値を書き換えてよい。例：settings = [("Ridge", 0.01, 2), ("Ridge", 10, 2), ("Ridge", 1000, 2), ("Lasso", 0.01, 2), ("Lasso", 1, 2)]
#  ③ 実行すると、この settings の各行が1試行として lab に記録される（前に試した試行は消えず、同じ設定は重複記録されない）
#  ④ 下の「経過の確認」セルで結果を見て、①②を書き換えて再実行する（これを何回も繰り返す）
#  ※ 15 回以上。Ridge と Lasso の両方、degree も動かすこと。
# 進め方の例： ①桁を広く振る（粗探索） → ②良さそうな桁の周辺を細かく（絞り込み） → ③model・degree を変えて確認
# 毎回「何を確かめる試行か」を hypothesis に書くこと（短すぎると report で警告される）
hypothesis = "まず alpha を桁で広く振り、Ridge と Lasso で良さそうな桁を掴む（degree=2）"
settings = [                       # (モデル, alpha, degree)
    ("Ridge", 0.01, 2), ("Ridge", 10, 2), ("Ridge", 1000, 2),
    ("Lasso", 0.01, 2), ("Lasso", 1, 2),
]
for name, alpha, degree in settings:
    run_trial(hypothesis, name, alpha, degree)

  #04 {'model': 'Ridge', 'alpha': 0.01, 'degree': 2}  cv_score=0.6569  cv_std=0.102  train_r2=0.92  n_nonzero=119/119  max_abs_coef=9.0
  #05 {'model': 'Ridge', 'alpha': 10, 'degree': 2}  cv_score=0.8296  cv_std=0.05  train_r2=0.89  n_nonzero=119/119  max_abs_coef=1.0  ★ベスト更新
  #06 {'model': 'Ridge', 'alpha': 1000, 'degree': 2}  cv_score=0.7399  cv_std=0.035  train_r2=0.776  n_nonzero=119/119  max_abs_coef=0.3
  #07 {'model': 'Lasso', 'alpha': 0.01, 'degree': 2}  cv_score=0.8254  cv_std=0.061  train_r2=0.893  n_nonzero=64/119  max_abs_coef=2.0
  #08 {'model': 'Lasso', 'alpha': 1, 'degree': 2}  cv_score=0.7710  cv_std=0.054  train_r2=0.774  n_nonzero=1/119  max_abs_coef=3.0


In [8]:
hypothesis = "Lasso は alpha=1 で非ゼロ係数がほぼ0（#8）。0.01〜1 の間を細かく見て、Ridge は 10〜1000 の間を見る"
settings = [("Lasso", 0.03, 2), ("Lasso", 0.1, 2), ("Lasso", 0.3, 2), ("Ridge", 30, 2), ("Ridge", 100, 2), ("Ridge", 300, 2)]
for name, alpha, degree in settings:
    run_trial(hypothesis, name, alpha, degree)

  #09 {'model': 'Lasso', 'alpha': 0.03, 'degree': 2}  cv_score=0.8374  cv_std=0.049  train_r2=0.87  n_nonzero=29/119  max_abs_coef=2.3  ★ベスト更新
  #10 {'model': 'Lasso', 'alpha': 0.1, 'degree': 2}  cv_score=0.8336  cv_std=0.057  train_r2=0.849  n_nonzero=13/119  max_abs_coef=3.3
  #11 {'model': 'Lasso', 'alpha': 0.3, 'degree': 2}  cv_score=0.8198  cv_std=0.059  train_r2=0.825  n_nonzero=6/119  max_abs_coef=3.4
  #12 {'model': 'Ridge', 'alpha': 30, 'degree': 2}  cv_score=0.8335  cv_std=0.045  train_r2=0.876  n_nonzero=119/119  max_abs_coef=0.7
  #13 {'model': 'Ridge', 'alpha': 100, 'degree': 2}  cv_score=0.8251  cv_std=0.045  train_r2=0.857  n_nonzero=119/119  max_abs_coef=0.6
  #14 {'model': 'Ridge', 'alpha': 300, 'degree': 2}  cv_score=0.8035  cv_std=0.043  train_r2=0.832  n_nonzero=119/119  max_abs_coef=0.4


In [9]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("cv_score", ascending=False).head(8))   # 上位8件
lab.plot(x="alpha", group=["model", "degree"], query="model != 'Linear'")   # alpha と CV R²、ベスト更新の推移
lab.report(min_trials=15)                                                    # 探索の自己診断

    trial   model   alpha  degree  cv_score  cv_std  train_r2 n_nonzero  max_abs_coef
8       9   Lasso    0.03       2    0.8374   0.049     0.870    29/119           2.3
9      10   Lasso    0.10       2    0.8336   0.057     0.849    13/119           3.3
11     12   Ridge   30.00       2    0.8335   0.045     0.876   119/119           0.7
4       5   Ridge   10.00       2    0.8296   0.050     0.890   119/119           1.0
6       7   Lasso    0.01       2    0.8254   0.061     0.893    64/119           2.0
12     13   Ridge  100.00       2    0.8251   0.045     0.857   119/119           0.6
0       1  Linear    0.00       1    0.8227   0.053     0.842     14/14           3.6
10     11   Lasso    0.30       2    0.8198   0.059     0.825     6/119           3.4
■ 試行回数: 14（目安 15 回以上） → まだ足りない
■ ベスト: trial#9 {'model': 'Lasso', 'alpha': 0.03, 'degree': 2}  cv_score=0.8374
■ model: 3 種類を試した
■ alpha: 11 種類を試した（範囲 0〜1000）
■ degree: 3 種類を試した（範囲 1〜3）
■ 仮説を書いた試行: 14/14
■ 前半の最高 0.8227 → 後半の最高 

In [10]:
hypothesis = "degree を 1 と 3 に変えると、正則化の最適な強さと CV R² はどう動くか"
settings = [("Ridge", 30, 1), ("Ridge", 30, 3), ("Ridge", 300, 3), ("Lasso", 0.1, 1), ("Lasso", 0.1, 3)]
for name, alpha, degree in settings:
    run_trial(hypothesis, name, alpha, degree)

  #15 {'model': 'Ridge', 'alpha': 30, 'degree': 1}  cv_score=0.8131  cv_std=0.048  train_r2=0.833  n_nonzero=14/14  max_abs_coef=2.9
  #16 {'model': 'Ridge', 'alpha': 30, 'degree': 3}  cv_score=0.7900  cv_std=0.072  train_r2=0.909  n_nonzero=679/679  max_abs_coef=0.5
  #17 {'model': 'Ridge', 'alpha': 300, 'degree': 3}  cv_score=0.8118  cv_std=0.045  train_r2=0.862  n_nonzero=679/679  max_abs_coef=0.2
  #18 {'model': 'Lasso', 'alpha': 0.1, 'degree': 1}  cv_score=0.8269  cv_std=0.058  train_r2=0.837  n_nonzero=6/14  max_abs_coef=3.6
  #19 {'model': 'Lasso', 'alpha': 0.1, 'degree': 3}  cv_score=0.8292  cv_std=0.054  train_r2=0.854  n_nonzero=22/679  max_abs_coef=3.3


In [11]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("cv_score", ascending=False).head(8))   # 上位8件
lab.plot(x="alpha", group=["model", "degree"], query="model != 'Linear'")   # alpha と CV R²、ベスト更新の推移
lab.report(min_trials=15)                                                    # 探索の自己診断

    trial  model   alpha  degree  cv_score  cv_std  train_r2 n_nonzero  max_abs_coef
8       9  Lasso    0.03       2    0.8374   0.049     0.870    29/119           2.3
9      10  Lasso    0.10       2    0.8336   0.057     0.849    13/119           3.3
11     12  Ridge   30.00       2    0.8335   0.045     0.876   119/119           0.7
4       5  Ridge   10.00       2    0.8296   0.050     0.890   119/119           1.0
18     19  Lasso    0.10       3    0.8292   0.054     0.854    22/679           3.3
17     18  Lasso    0.10       1    0.8269   0.058     0.837      6/14           3.6
6       7  Lasso    0.01       2    0.8254   0.061     0.893    64/119           2.0
12     13  Ridge  100.00       2    0.8251   0.045     0.857   119/119           0.6
■ 試行回数: 19（目安 15 回以上） → OK
■ ベスト: trial#9 {'model': 'Lasso', 'alpha': 0.03, 'degree': 2}  cv_score=0.8374
■ model: 3 種類を試した
■ alpha: 11 種類を試した（範囲 0〜1000）
■ degree: 3 種類を試した（範囲 1〜3）
■ 仮説を書いた試行: 19/19
■ 前半の最高 0.8296 → 後半の最高 0.8292


In [12]:
hypothesis = "Ridge と Lasso の中間の ElasticNet は両方の良い所を取れるか"
settings = [("ElasticNet", 0.1, 2), ("ElasticNet", 0.03, 2), ("ElasticNet", 0.3, 2)]
for name, alpha, degree in settings:
    run_trial(hypothesis, name, alpha, degree)

  #20 {'model': 'ElasticNet', 'alpha': 0.1, 'degree': 2}  cv_score=0.8382  cv_std=0.047  train_r2=0.858  n_nonzero=30/119  max_abs_coef=1.2  ★ベスト更新
  #21 {'model': 'ElasticNet', 'alpha': 0.03, 'degree': 2}  cv_score=0.8358  cv_std=0.051  train_r2=0.878  n_nonzero=50/119  max_abs_coef=1.3
  #22 {'model': 'ElasticNet', 'alpha': 0.3, 'degree': 2}  cv_score=0.8216  cv_std=0.054  train_r2=0.835  n_nonzero=20/119  max_abs_coef=1.1


In [13]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("cv_score", ascending=False).head(8))   # 上位8件
lab.plot(x="alpha", group=["model", "degree"], query="model != 'Linear'")   # alpha と CV R²、ベスト更新の推移
lab.report(min_trials=15)                                                    # 探索の自己診断

    trial       model  alpha  degree  cv_score  cv_std  train_r2 n_nonzero  max_abs_coef
19     20  ElasticNet   0.10       2    0.8382   0.047     0.858    30/119           1.2
8       9       Lasso   0.03       2    0.8374   0.049     0.870    29/119           2.3
20     21  ElasticNet   0.03       2    0.8358   0.051     0.878    50/119           1.3
9      10       Lasso   0.10       2    0.8336   0.057     0.849    13/119           3.3
11     12       Ridge  30.00       2    0.8335   0.045     0.876   119/119           0.7
4       5       Ridge  10.00       2    0.8296   0.050     0.890   119/119           1.0
18     19       Lasso   0.10       3    0.8292   0.054     0.854    22/679           3.3
17     18       Lasso   0.10       1    0.8269   0.058     0.837      6/14           3.6
■ 試行回数: 22（目安 15 回以上） → OK
■ ベスト: trial#20 {'model': 'ElasticNet', 'alpha': 0.1, 'degree': 2}  cv_score=0.8382
■ model: 4 種類を試した
■ alpha: 11 種類を試した（範囲 0〜1000）
■ degree: 3 種類を試した（範囲 1〜3）
■ 仮説を書いた試行: 2

In [14]:
# === 最終評価：探索を終えたら「1回だけ」実行する（2回目はエラー。テストを見て設定を変えるのはリーク）===
FINAL = {"model": "Lasso", "alpha": 0.1, "degree": 1}   # 1標準偏差ルールで選び直した（理由は解答欄 2-d）
# FINAL = {"model": "Ridge", "alpha": 100, "degree": 2}   # ← 選び直すならここ（理由は解答欄へ）

# ★あなたが書く★：FINAL の設定でモデルを作り、訓練データ全体で学習して final_model に入れる（1行）
#   ヒント: make_model(FINAL["model"], FINAL["alpha"], FINAL["degree"]).fit(訓練の説明変数, 訓練の目的変数)
final_model = make_model(FINAL["model"], FINAL["alpha"], FINAL["degree"]).fit(X_train, y_train)

lab.final_test(lambda: final_model.score(X_test, y_test))
print("採用した設定:", FINAL)

最終テストスコア = 0.8003   （検証ベスト = 0.8382）
採用した設定: {'model': 'Lasso', 'alpha': 0.1, 'degree': 1}


In [15]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd

# (2-a) 実験ログ（自動出力。全試行が出力に残る。15回以上・Ridge/Lasso 両方・degree も変えること）
experiment_log = lab.df()
print(experiment_log.drop(columns="hypothesis").to_string())
lab.save()   # 手元にも CSV で保存（q9_regression_log.csv）

# (2-b) 探索の進め方：各段階で何を見て次の試行を決めたか。粗探索→絞り込みの流れを、試行番号 #n を引用して書く
answer_2_b = """
【粗探索 #4〜#8】degree=2 に固定して alpha を 0.01 / 10 / 1000（Ridge）、0.01 / 1（Lasso）と桁で振った。Ridge は 0.01 で CV 0.657（過学習）→ 10 で 0.830 → 1000 で 0.740（強すぎ）と山型。Lasso は alpha=1 で非ゼロ係数が 1/119 になり CV 0.771 に下がった（#8）。
【絞り込み #9〜#14】この結果から Lasso は 0.01〜1 の間（0.03/0.1/0.3）、Ridge は 10〜1000 の間（30/100/300）を細かく見た。Lasso 0.03 が 0.8374（#9）、Ridge 30 が 0.8335（#12）で最良付近。
【軸を変えて確認 #15〜#19】degree を 1 と 3 に変え、Ridge 30 では degree=1: 0.813、degree=3: 0.790 と degree=2 が最良。Lasso 0.1 では degree=1: 0.827、2: 0.834、3: 0.829 とほぼ差が無い。
【中間の手法 #20〜#22】ElasticNet 0.1 が 0.8382 と全体で最良（#20）。ここで上位の差が cv_std（約 0.05）より遥かに小さいと分かり、探索を打ち切った。
"""

# (2-c) 仮説が外れた試行を1つ以上取り上げ、なぜ外れたか・何を学んだか（試行番号 #n を引用）
answer_2_c = """
仮説「Lasso alpha=1 が丁度良い」は外れた（#8: 非ゼロ係数 1/119、CV 0.771）。標準化済みで目的変数が 0〜20 点のこのデータでは、alpha=1 の L1 は「ほぼ G2 だけ」を使うほど強すぎた。Lasso の実質的な最適 alpha は 0.03〜0.1 で、予想より 1 桁以上小さい。
もう 1 つ、仮説「degree を上げれば表現力が上がり CV も上がる」も外れた（#2, #3, #16）。正則化なしでは degree=3 で CV R²=-1.14、Ridge 30 でも 0.790 で degree=2 より低い。特徴量が増えるだけで、G1・G2 以外に追加の情報が乏しいから。
"""

# (2-d) 最良設定（model / alpha / degree と cv_score ± cv_std）。上位候補との差は cv_std に対して意味があるか。最終的に採用した設定とその理由
answer_2_d = """
CV 最良は #20 ElasticNet(alpha=0.1, degree=2) の 0.8382 ± 0.047。上位 (#9 Lasso 0.8374, #10 0.8336, #12 Ridge 0.8335, #5 0.8296, #1 Linear degree=1 の 0.8227) との差は 0.01 程度で、cv_std（0.045〜0.06）よりずっと小さい → 「差がある」とは言えない。
1 標準偏差ルール（最良 0.8382 − 0.047 = 0.791 以上の中で最も単純なもの）を適用すると、Linear degree=1 (0.823) や Lasso(alpha=0.1, degree=1)（0.8269, 非ゼロ係数 6/14）が候補になる。後者は正則化つきで、使う特徴量が 6 個に絞られ、解釈しやすい。そこで最終的には Lasso(alpha=0.1, degree=1) を採用した。
"""

# (2-e) 最終テストの結果は CV の値と比べてどうだったか。この差をどう解釈するか（報告する性能はどちらか）
answer_2_e = """
最終テスト R² = 0.8003、CV は 0.8269（採用設定）／探索ベストは 0.8382。テストは CV より 0.02〜0.04 低かったが、cv_std（0.058）の範囲内で、想定内の差。探索を重ねた検証スコアは楽観的になる（選択バイアス）ので、報告する性能は「テストの 0.80」の方（1回だけ使った値）。
"""

# (2-f) 実験前の予測（prediction_2_*）と実際の最良設定を比べる。どこがズレた？ その原因は？ 予測の根拠のうち、正しかった知識／間違っていた知識は？
answer_2_f = """
予測は Lasso / alpha=1 / degree=2。実際の最良は ElasticNet / alpha=0.1 / degree=2（Lasso 単独でも alpha=0.03〜0.1）。degree=2 と「Lasso 系が良い」は当たったが、alpha は 1 桁ズレた。原因は、alpha の目安を「標準化したから 1 くらい」と決めつけ、Lasso の目的関数が RSS/(2n) と n で割られている点や目的変数のスケール（0〜20 点）を考えなかったこと。
正しかった知識：過学習には正則化が効く／degree を上げすぎると CV が崩れる。間違っていた知識：最適な alpha の桁（LLM の回答も同様にずれていた）。
"""

# (2-g) 考察1：alpha と CV R² の関係（ちょうど良い alpha はあったか）
reflection1 = """
alpha が小さすぎる（Ridge 0.01: CV 0.657、#4）と過学習で CV が低く、大きすぎる（Ridge 1000: 0.740、#6／Lasso 1: 0.771、#8）と係数が縮みすぎて未学習になる。Ridge は 10〜100 付近（#5 0.830, #12 0.8335, #13 0.825）が山で、両側で下がる「ちょうど良い alpha」があった。Lasso は 0.03〜0.1（#9, #10）。
"""

# (2-h) 考察1'：ベストが探索範囲の端にあったとき何を意味するか／Lasso で alpha を上げすぎると何が起きるか
reflection1b = """
ベストが探索範囲の端にあるなら、その先にもっと良い設定がある可能性があり、範囲を広げて確かめる必要がある。今回は Ridge の最良（30）が探索範囲 0.01〜1000 の内側（山型）だったので、範囲は妥当だった。一方、Lasso で alpha を上げすぎると（alpha=1: 非ゼロ係数 1/119）ほぼ全係数が 0 になり、予測が平均値＋ごく少数の特徴量だけになって R² が下がる（さらに大きいと R²≈0 に張り付く）。
"""

# (2-i) 考察2：L1 と L2 の違い（なぜ Lasso は係数がゼロになる＝特徴量選択になるのか）
reflection2 = """
L1（Σ|w|）のペナルティは、w=0 付近でも勾配の大きさが一定（傾き ±α）なので、損失の勾配がその大きさを超えない特徴量の係数は「0 に留まる」のが最適になる（0 の点が尖っている）。L2（Σw²）は w が小さいほど勾配も小さく（2αw）、0 に近づくほど縮める力が弱まるため、係数は小さくはなるが 0 にはならない。実際、Lasso の非ゼロ係数は alpha=0.01: 64 → 0.03: 29 → 0.1: 13 → 0.3: 6 → 1: 1 と減ったが、Ridge は常に 119/119 だった。「Lasso が特徴量選択になる」とは、不要な特徴量の係数が厳密に 0 になり、モデルが使う変数が自動的に絞られるという意味。
"""

# (2-j) 考察2'：degree=1 の Linear と最良設定の比較。設計判断1（A〜F）の選択は最良だったか
reflection2b = """
degree=1 の Linear は CV 0.8227 で、正則化した最良設定 0.8382 との差は 0.016 と、cv_std（0.05）より小さい → 有意な改善とは言えない。設計判断1で選んだ (B) Ridge は #12 で 0.8335 と Linear 以上だったが、差は誤差範囲。(E) degree を下げる（Linear degree=1）が手軽で同等の結果だった。つまりこのデータは G1・G2 で説明でき、多項式展開で増やした特徴量に新しい情報がほとんど無かった。正則化は「増やしすぎた特徴量による過学習を打ち消す」役割は果たしたが、「最初から増やさない」方がずっと手軽だった。
"""

    trial       model    alpha  degree  cv_score  cv_std  train_r2 n_nonzero  max_abs_coef
0       1      Linear     0.00       1    0.8227   0.053     0.842     14/14           3.6
1       2      Linear     0.00       2    0.6443   0.106     0.920   119/119           9.6
2       3      Linear     0.00       3   -1.1404   0.773     1.000   679/679          10.3
3       4       Ridge     0.01       2    0.6569   0.102     0.920   119/119           9.0
4       5       Ridge    10.00       2    0.8296   0.050     0.890   119/119           1.0
5       6       Ridge  1000.00       2    0.7399   0.035     0.776   119/119           0.3
6       7       Lasso     0.01       2    0.8254   0.061     0.893    64/119           2.0
7       8       Lasso     1.00       2    0.7710   0.054     0.774     1/119           3.0
8       9       Lasso     0.03       2    0.8374   0.049     0.870    29/119           2.3
9      10       Lasso     0.10       2    0.8336   0.057     0.849    13/119           3.3

## 問題3　ロジスティック回帰（正則化つき分類）を説明する　【説明】
***

### 回帰から分類へ

「点数を予測する（回帰）」から「合格/不合格を予測する（分類）」に問題を変換する。機械学習の実務では，最終的な判断が「Yes/No」「合格/不合格」「異常/正常」であることが多く，この変換はよく行われる。

### ロジスティック回帰とは

名前に「回帰」が入っているが，実際は**分類モデル**だ。内部では：

1. 線形結合を計算: z = w₁x₁ + w₂x₂ + ... + b
2. シグモイド関数で確率に変換: P(y=1) = 1 / (1 + e^(-z))
3. P > 0.5 なら 1（合格），P ≤ 0.5 なら 0（不合格）と予測

シグモイド関数は出力を必ず [0, 1] の範囲に収めるため，確率として解釈できる。

### 説明変数の選択について

今回使う変数の意味：
- `G1`, `G2`: 1〜2学期の成績（最も予測力が高いはず）
- `studytime`: 週の勉強時間（1=2時間未満 〜 4=10時間以上）
- `failures`: 過去の留年回数

> **考えてみよう**: なぜ `G1`, `G2` だけでなく `studytime` や `failures` も加えるのか？単純に成績だけでなく「学習習慣」や「過去の失敗」が最終成績の予測に寄与するかを確かめたいからだ。

### 課題

ロジスティック回帰も実は**正則化つきのモデル**です（`LogisticRegression` はデフォルトで L2 正則化がかかっています）。ここでは「合格/不合格」を予測する分類モデルの**構築コードの中身を理解**します。

下のコードセルは **完成形**です。**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたらセルを実行し、エラーなく正解率が出ることを確認してください。

説明を書くときは、次の問いを意識してください：

- `penalty="l2"` は何にペナルティをかけているか？（問題2の Ridge と同じ考え方）
- `C` は正則化の強さと**どういう関係**か？（`alpha` と逆向きだったことを思い出す）
- `solver`（最適化アルゴリズム）と `max_iter` は何を決めているか？なぜ `max_iter` を大きくするのか？

> **設計判断（一言で）**: もし `C` を `1.0` から `0.001`（とても小さい値）に変えたら、係数とモデルの複雑さはどうなると思いますか？ 解答用コードセルに1〜2文で書いてください。

In [16]:
# 各行の「# 説明:」に自分の言葉で意味を書いてください（コードは変更しない）。
# （コードは AI に書かせても構いませんが、この説明欄は AI の出力を貼らず、理解した内容を自分の言葉で書くこと）

y_clf = (df["G3"] >= 10).astype(int)                  # 説明: 最終成績 G3 が 10 点以上なら 1（合格），未満なら 0（不合格）という2値のラベルに変換している．回帰問題を分類問題に置き換える操作．
clf_cols = ["G1", "G2", "studytime", "failures"]
Xc = df[clf_cols].values

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, y_clf, test_size=0.2, random_state=0, stratify=y_clf  # 説明: 訓練/テストに分けるとき，合格・不合格の比率を両方で元データと同じに保つ（層化抽出）．偏った分割で評価がぶれるのを防ぐ．
)

clf_scaler = StandardScaler()
Xc_train_s = clf_scaler.fit_transform(Xc_train)       # 説明: 訓練データだけで平均・標準偏差を計算（fit）し，その値で標準化（transform）する．テストの情報を使わないためで，訓練データでのみ fit する．
Xc_test_s = clf_scaler.transform(Xc_test)             # 説明: テストは，訓練データで求めた平均・標準偏差を使って変換だけする．テストで fit し直すとテストの統計量が学習に漏れる（データリーク）．

logreg = LogisticRegression(
    penalty="l2",     # 説明: ペナルティ（正則化）の種類．L2 は係数の二乗和 Σw² を損失に足して，係数が大きくなりすぎるのを防ぐ（Ridge と同じ考え方）．
    C=1.0,            # 説明: C は正則化の強さの「逆数」．C が小さいほど正則化が強く係数が小さくなり，大きいほど弱くなる．alpha とは逆向き．
    solver="lbfgs",   # 説明: 損失を最小にする係数を見つける最適化アルゴリズム（準ニュートン法）の種類を決める．答えは同じでも，速さや使える penalty が異なる．
    max_iter=1000,    # 説明: 最適化の反復回数の上限．既定の 100 回では収束しないことがあり，収束前に打ち切られる警告を防ぐために大きくする．
)
logreg.fit(Xc_train_s, yc_train)                      # 説明: 標準化済みの訓練データ（説明変数と合否ラベル）でロジスティック回帰の係数を学習する．
acc = accuracy_score(yc_test, logreg.predict(Xc_test_s))  # 説明: テストデータの合否を予測し，実際の正解と一致した割合（正解率）を計算する．
print("テスト正解率 =", round(acc, 4))


テスト正解率 = 0.9114


In [17]:
# === ✍️ 問題3 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (3-a) `penalty="l2"` は何にペナルティをかけているか
answer_3_a = """
係数の二乗和 Σw² に比例したペナルティを損失関数に加える。係数が大きくなりすぎるとペナルティが大きくなるので、係数を小さく（0 に近く）保つ方向に働く。
"""

# (3-b) `C` は正則化の強さとどういう関係か（`alpha` と逆向きだったことを踏まえて）
answer_3_b = """
C は正則化の強さの「逆数」。C が大きいと正則化は弱く、小さいと強い。Ridge/Lasso の alpha は大きいほど強いので、向きが逆。
"""

# (3-c) `solver` と `max_iter` は何を決めているか
answer_3_c = """
solver は損失を最小化する係数を探す最適化アルゴリズムを決める（lbfgs は準ニュートン法で L2 向き）。max_iter はその反復回数の上限で、足りないと収束前に止まって警告が出るので、既定の 100 より大きい 1000 にしている。
"""

# (3-d) 設計判断：`C` を `1.0` から `0.001` に変えると、係数とモデルの複雑さはどうなるか（1〜2文）
answer_3_d = """
C を 0.001 にすると正則化がとても強くなり、係数は 0 に近づいてモデルは非常に単純になる（未学習になり、ほぼ「多数派のクラスを予測する」だけになりうる）。
"""



## 問題4　ロジスティック回帰の最適設定を自分で探す（分類）　【探索】
***

### 課題

「合格（`G3>=10`）/ 不合格」を予測します。問題3のように `G1`・`G2`（前の成績）を使うとほぼ何をしても高精度になり違いが見えないので、この問題では **`G1`・`G2` を外し**、学習習慣・家庭環境・生活だけで予測する難しい設定にします。

**CV 正解率が最も高い設定を、自分で探してください。** 探す軸は次の4つです。

| 軸 | 選択肢 | 注意点 |
|---|---|---|
| `penalty` | `"l1"` / `"l2"` | l1 は不要な特徴量の係数を0にする |
| `C` | 正の実数（**大きいほど正則化が弱い**。alpha と逆向き） | 対数スケールで考える |
| `degree` | 1 / 2 / 3 | 特徴量が増えて過学習しやすくなる |
| `groups` | 特徴量グループの組み合わせ | 下の3グループから選ぶ。**どの情報が予測に効くかを自分で考える** |

```
学習習慣: studytime, failures, absences
家庭環境: Medu, Fedu
生活    : age, traveltime, freetime, goout, Dalc, Walc, health
```

### 手順（問題2と同じ）

1. 先に **特徴量グループの重要度の予測** を書く（実験後に書き直さない）。
2. `settings` を書き換えて何度も試す（**12回以上**、`groups` を変える試行を含める）。
3. 経過確認セルで、頭打ち・端・グループごとの効きを確認する。
4. 最終評価セルを **1回だけ** 実行する。
5. 解答セルに書く。

> **考察3**: C を小さくすると訓練正解率は、C を大きくすると訓練と CV の差はどう動きましたか？ どのあたりが「ちょうど良い C」でしたか？
>
> **考察4**: 問題2の `alpha` と，この問題の `C` は**逆向き**。「`alpha` を大きくする」と「`C` を大きくする」は、正則化の強さの観点でそれぞれどちらに向かうか。
>
> **考察5**: なぜ C の選択にテストデータを使ってはいけないのか？ 最終テスト正解率と CV 正解率が違ったとき、どちらを「性能の見積もり」として報告すべきか？ また「全員合格と予測」のベースライン（≈0.67）と比べて、このモデルはどれだけ役に立っているか？
>
> **考察6（知識の使い方）**: `groups` を変えた試行から、どの情報が予測に効いていると言えますか？ その結果は「因果」と言えますか？（例：`failures` が効く＝留年させると成績が上がる／下がる、と言えるか）

In [18]:
# === ✍️ 問題4-0 実験前の予測（実験を始める前に書く。実験後に書き直さない）===

# 「学習習慣」「家庭環境」「生活」のうち、合否の予測に最も効くと思う順に並べる（例："学習習慣 > 生活 > 家庭環境"）
prediction_4_ranking = "学習習慣 > 生活 > 家庭環境"
# そう予測した理由
prediction_4_reason = """
成績（合否）は、勉強時間・過去の落第・欠席といった学習習慣に最も直接的に関係すると考えた。次に、飲酒（Dalc/Walc）や外出（goout）など生活習慣が影響し、親の学歴（Medu/Fedu）は間接的なので最も弱いと予想した。
"""

In [19]:
# === 完成済みコード（★の箇所だけ自分で書く）===
y_clf = (df["G3"] >= 10).astype(int)                  # 合格=1 / 不合格=0

FEATURE_GROUPS = {
    "学習習慣": ["studytime", "failures", "absences"],
    "家庭環境": ["Medu", "Fedu"],
    "生活":     ["age", "traveltime", "freetime", "goout", "Dalc", "Walc", "health"],
}
all_cols = [c for cols in FEATURE_GROUPS.values() for c in cols]
X4_train, X4_test, y4_train, y4_test = train_test_split(
    df[all_cols], y_clf, test_size=0.2, random_state=0, stratify=y_clf
)
print(f"ベースライン（全員合格と予測）の正解率 = {max(y4_test.mean(), 1 - y4_test.mean()):.3f}")

cv5s = StratifiedKFold(5, shuffle=True, random_state=0)
lab4 = Lab("q9_logreg", score_name="cv_acc")

def make_clf(penalty, C, degree):
    # ★あなたが書く★：多項式展開 → 標準化 → ロジスティック回帰 をつなぐ Pipeline を返す（1行）
    #   ヒント: make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(),
    #                        LogisticRegression(penalty=penalty, C=C, solver="liblinear", max_iter=5000))
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(),
                         LogisticRegression(penalty=penalty, C=C, solver="liblinear", max_iter=5000))

def run_trial4(hypothesis, penalty, C, degree, groups):
    cols = [c for g in groups for c in FEATURE_GROUPS[g]]
    params = {"penalty": penalty, "C": C, "degree": degree, "groups": "+".join(groups)}
    if lab4.seen(params):
        print(f"  (skip) 既に試した設定です: {params}"); return
    m = make_clf(penalty, C, degree)
    scores = cross_val_score(m, X4_train[cols], y4_train, cv=cv5s)     # 訓練データ内の5分割CV
    m.fit(X4_train[cols], y4_train)
    lab4.log(hypothesis, params, scores.mean(),
             cv_std=round(scores.std(), 3), train_acc=round(m.score(X4_train[cols], y4_train), 3))

# === ★ここを変えて実験する★：仮説を書き、settings を書き換えて何度でも実行 ===
# 【変更のしかた】
#  ① hypothesis … 今回の試行で「何を確かめたいか」を1文で書き換える（例：「alpha を10倍にすると CV は上がるはず」）。空・短すぎると report で警告される
#  ② settings   … 試したい設定を「1行に1つのタプル」で並べる。タプルの並び順は (penalty, C, degree, groups)
#       penalty … "l2" / "l1"
#       C       … 正則化の強さの逆数（大きいほど弱い）。桁で振る（0.001, 0.01, …, 100）
#       degree  … 多項式の次数 1 / 2 / 3
#       groups  … 使う特徴量グループを、名前の「タプル」で指定："学習習慣" / "家庭環境" / "生活"。1つだけのときは末尾にカンマ：("学習習慣",)
#     行を足す／消す／数値を書き換えてよい。例：settings = [("l2", 0.001, 2, ("学習習慣", "家庭環境", "生活")), ("l2", 0.1, 2, ("学習習慣", "家庭環境", "生活")), ("l2", 100, 2, ("学習習慣", "家庭環境", "生活"))]
#  ③ 実行すると、この settings の各行が1試行として lab に記録される（前に試した試行は消えず、同じ設定は重複記録されない）
#  ④ 下の「経過の確認」セルで結果を見て、①②を書き換えて再実行する（これを何回も繰り返す）
#  ※ 12 回以上。groups を変えた試行を含めること。
hypothesis = "全グループを使い、C を桁で広く振って l2 の傾向を掴む"
settings = [                       # (penalty, C, degree, groups)
    ("l2", 0.001, 2, ("学習習慣", "家庭環境", "生活")),
    ("l2", 0.1,   2, ("学習習慣", "家庭環境", "生活")),
    ("l2", 100,   2, ("学習習慣", "家庭環境", "生活")),
]
for penalty, C, degree, groups in settings:
    run_trial4(hypothesis, penalty, C, degree, groups)

ベースライン（全員合格と予測）の正解率 = 0.671
  #01 {'penalty': 'l2', 'C': 0.001, 'degree': 2, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7090  cv_std=0.02  train_acc=0.734  ★ベスト更新
  #02 {'penalty': 'l2', 'C': 0.1, 'degree': 2, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7215  cv_std=0.024  train_acc=0.763  ★ベスト更新
  #03 {'penalty': 'l2', 'C': 100, 'degree': 2, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.6422  cv_std=0.045  train_acc=0.823


In [20]:
hypothesis = "l1 でも同じ C で試す。強い l1 では係数が全部0になるはず"
settings = [("l1", 0.01, 2, ("学習習慣", "家庭環境", "生活")), ("l1", 0.1, 2, ("学習習慣", "家庭環境", "生活")), ("l1", 1, 2, ("学習習慣", "家庭環境", "生活"))]
for penalty, C, degree, groups in settings:
    run_trial4(hypothesis, penalty, C, degree, groups)

  #04 {'penalty': 'l1', 'C': 0.01, 'degree': 2, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.3291  cv_std=0.006  train_acc=0.329
  #05 {'penalty': 'l1', 'C': 0.1, 'degree': 2, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7404  cv_std=0.028  train_acc=0.756  ★ベスト更新
  #06 {'penalty': 'l1', 'C': 1, 'degree': 2, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7027  cv_std=0.032  train_acc=0.778


In [21]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab4.df().drop(columns="hypothesis").sort_values("cv_acc", ascending=False).head(8))
lab4.plot(x="C", group=["penalty", "degree"])          # C と CV 正解率、ベスト更新の推移
lab4.report(min_trials=12)

   trial penalty        C  degree        groups  cv_acc  cv_std  train_acc
4      5      l1    0.100       2  学習習慣+家庭環境+生活  0.7404   0.028      0.756
1      2      l2    0.100       2  学習習慣+家庭環境+生活  0.7215   0.024      0.763
0      1      l2    0.001       2  学習習慣+家庭環境+生活  0.7090   0.020      0.734
5      6      l1    1.000       2  学習習慣+家庭環境+生活  0.7027   0.032      0.778
2      3      l2  100.000       2  学習習慣+家庭環境+生活  0.6422   0.045      0.823
3      4      l1    0.010       2  学習習慣+家庭環境+生活  0.3291   0.006      0.329
■ 試行回数: 6（目安 12 回以上） → まだ足りない
■ ベスト: trial#5 {'penalty': 'l1', 'C': 0.1, 'degree': 2, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7404
■ penalty: 2 種類を試した
■ C: 5 種類を試した（範囲 0.001〜100）
■ degree: 1 種類を試した
■ groups: 1 種類を試した
■ 仮説を書いた試行: 6/6
■ 前半の最高 0.7215 → 後半の最高 0.7404


In [22]:
hypothesis = "degree=1 に下げると過学習が減って CV が上がるか（l2 と l1）"
settings = [("l2", 0.1, 1, ("学習習慣", "家庭環境", "生活")), ("l2", 1, 1, ("学習習慣", "家庭環境", "生活")), ("l1", 0.1, 1, ("学習習慣", "家庭環境", "生活")), ("l1", 1, 1, ("学習習慣", "家庭環境", "生活"))]
for penalty, C, degree, groups in settings:
    run_trial4(hypothesis, penalty, C, degree, groups)

  #07 {'penalty': 'l2', 'C': 0.1, 'degree': 1, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7184  cv_std=0.012  train_acc=0.734
  #08 {'penalty': 'l2', 'C': 1, 'degree': 1, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7152  cv_std=0.017  train_acc=0.737
  #09 {'penalty': 'l1', 'C': 0.1, 'degree': 1, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7405  cv_std=0.016  train_acc=0.741  ★ベスト更新
  #10 {'penalty': 'l1', 'C': 1, 'degree': 1, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7216  cv_std=0.015  train_acc=0.741


In [23]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab4.df().drop(columns="hypothesis").sort_values("cv_acc", ascending=False).head(8))
lab4.plot(x="C", group=["penalty", "degree"])          # C と CV 正解率、ベスト更新の推移
lab4.report(min_trials=12)

   trial penalty      C  degree        groups  cv_acc  cv_std  train_acc
8      9      l1  0.100       1  学習習慣+家庭環境+生活  0.7405   0.016      0.741
4      5      l1  0.100       2  学習習慣+家庭環境+生活  0.7404   0.028      0.756
9     10      l1  1.000       1  学習習慣+家庭環境+生活  0.7216   0.015      0.741
1      2      l2  0.100       2  学習習慣+家庭環境+生活  0.7215   0.024      0.763
6      7      l2  0.100       1  学習習慣+家庭環境+生活  0.7184   0.012      0.734
7      8      l2  1.000       1  学習習慣+家庭環境+生活  0.7152   0.017      0.737
0      1      l2  0.001       2  学習習慣+家庭環境+生活  0.7090   0.020      0.734
5      6      l1  1.000       2  学習習慣+家庭環境+生活  0.7027   0.032      0.778
■ 試行回数: 10（目安 12 回以上） → まだ足りない
■ ベスト: trial#9 {'penalty': 'l1', 'C': 0.1, 'degree': 1, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7405
■ penalty: 2 種類を試した
■ C: 5 種類を試した（範囲 0.001〜100）
■ degree: 2 種類を試した
■ groups: 1 種類を試した
■ 仮説を書いた試行: 10/10
■ 前半の最高 0.7215 → 後半の最高 0.7405


In [24]:
hypothesis = "グループを1つずつ外して、どの情報が効くか調べる（l2, C=0.1, degree=1）"
settings = [("l2", 0.1, 1, ("学習習慣",)), ("l2", 0.1, 1, ("家庭環境",)), ("l2", 0.1, 1, ("生活",)), ("l2", 0.1, 1, ("学習習慣", "家庭環境")), ("l2", 0.1, 1, ("学習習慣", "生活"))]
for penalty, C, degree, groups in settings:
    run_trial4(hypothesis, penalty, C, degree, groups)

  #11 {'penalty': 'l2', 'C': 0.1, 'degree': 1, 'groups': '学習習慣'}  cv_acc=0.7405  cv_std=0.019  train_acc=0.731
  #12 {'penalty': 'l2', 'C': 0.1, 'degree': 1, 'groups': '家庭環境'}  cv_acc=0.6709  cv_std=0.006  train_acc=0.671
  #13 {'penalty': 'l2', 'C': 0.1, 'degree': 1, 'groups': '生活'}  cv_acc=0.6488  cv_std=0.022  train_acc=0.684
  #14 {'penalty': 'l2', 'C': 0.1, 'degree': 1, 'groups': '学習習慣+家庭環境'}  cv_acc=0.7216  cv_std=0.023  train_acc=0.725
  #15 {'penalty': 'l2', 'C': 0.1, 'degree': 1, 'groups': '学習習慣+生活'}  cv_acc=0.7247  cv_std=0.017  train_acc=0.744


In [25]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab4.df().drop(columns="hypothesis").sort_values("cv_acc", ascending=False).head(8))
lab4.plot(x="C", group=["penalty", "degree"])          # C と CV 正解率、ベスト更新の推移
lab4.report(min_trials=12)

    trial penalty    C  degree        groups  cv_acc  cv_std  train_acc
8       9      l1  0.1       1  学習習慣+家庭環境+生活  0.7405   0.016      0.741
10     11      l2  0.1       1          学習習慣  0.7405   0.019      0.731
4       5      l1  0.1       2  学習習慣+家庭環境+生活  0.7404   0.028      0.756
14     15      l2  0.1       1       学習習慣+生活  0.7247   0.017      0.744
9      10      l1  1.0       1  学習習慣+家庭環境+生活  0.7216   0.015      0.741
13     14      l2  0.1       1     学習習慣+家庭環境  0.7216   0.023      0.725
1       2      l2  0.1       2  学習習慣+家庭環境+生活  0.7215   0.024      0.763
6       7      l2  0.1       1  学習習慣+家庭環境+生活  0.7184   0.012      0.734
■ 試行回数: 15（目安 12 回以上） → OK
■ ベスト: trial#9 {'penalty': 'l1', 'C': 0.1, 'degree': 1, 'groups': '学習習慣+家庭環境+生活'}  cv_acc=0.7405
■ penalty: 2 種類を試した
■ C: 5 種類を試した（範囲 0.001〜100）
■ degree: 2 種類を試した
■ groups: 6 種類を試した
■ 仮説を書いた試行: 15/15
■ 前半の最高 0.7404 → 後半の最高 0.7405


In [26]:
# === 最終評価：探索を終えたら「1回だけ」実行する ===
FINAL4 = {"penalty": "l2", "C": 0.1, "degree": 1, "groups": "学習習慣"}   # CV 最良(#9)と同点でより単純な設定に選び直した（理由は解答欄 4-c）
cols_f = [c for g in FINAL4["groups"].split("+") for c in FEATURE_GROUPS[g]]
final4 = make_clf(FINAL4["penalty"], FINAL4["C"], FINAL4["degree"]).fit(X4_train[cols_f], y4_train)
lab4.final_test(lambda: final4.score(X4_test[cols_f], y4_test))
print("採用した設定:", FINAL4)

最終テストスコア = 0.6835   （検証ベスト = 0.7405）
採用した設定: {'penalty': 'l2', 'C': 0.1, 'degree': 1, 'groups': '学習習慣'}


In [27]:
# === ✍️ 問題4 解答（採点対象）===
import pandas as pd

# (4-a) 実験ログ（自動出力。12回以上・l1/l2 の両方・groups を変えた試行を含めること）
experiment_log = lab4.df()
print(experiment_log.drop(columns="hypothesis").to_string())
lab4.save()

# (4-b) 探索の進め方：各段階で何を見て次の試行を決めたか（試行番号 #n を引用）／仮説が外れた試行とそこから学んだこと
answer_4_b = """
【#1〜#3】全グループ・degree=2 で l2 の C を 0.001 / 0.1 / 100 と振り、0.709 → 0.7215 → 0.642（C=100 は過学習：train 0.823 / cv 0.642）と山型。
【#4〜#6】l1 でも同様に。C=0.01 で CV=0.3291（#4）と極端に低い ── 仮説「弱い正則化から強い正則化まで滑らかに変わる」は外れた。C=0.1 で 0.7404（#5）。
【#7〜#10】degree=1 に下げると cv_std が小さくなり（0.012〜0.017）、l1 C=0.1 が 0.7405（#9）と最良。
【#11〜#15】グループを外す実験。学習習慣だけ（#11）で 0.7405 と全部入りと同点。家庭環境だけ 0.6709（#12）、生活だけ 0.6488（#13）。
#4 の仮説外れ（学んだこと）：C=0.01 の l1 では、liblinear が切片にも L1 ペナルティをかけるため、係数も切片も 0 になり、全員に同じクラス（この場合は不合格）を予測して正解率 0.329（＝不合格の比率）になった。「正解率がベースラインより極端に低い＝全員同じ予測」を疑うべき。
"""

# (4-c) 最良設定（penalty / C / degree / groups と cv_acc ± cv_std）と最終テスト正解率。ベースライン（≈0.67）と比べてどれだけ役に立つか
answer_4_c = """
CV 最良は #9 l1 / C=0.1 / degree=1 / 全グループ（cv_acc 0.7405 ± 0.016）。ただし #11（l2, C=0.1, degree=1, 学習習慣のみ）も 0.7405 ± 0.019 と同点なので、使う特徴量が 3 個で済む後者を採用した。最終テスト正解率は 0.6835。全員合格と予測するベースライン（≈0.671）より 0.012 ポイントしか高くなく、この設定（G1・G2 なし）では実用的な予測はほとんど出来ていない。CV（0.74）とテスト（0.68）の差は cv_std（0.019）の 3 倍あり、探索での楽観バイアスと、テストが 79 件と小さいこと（1 件で 0.013 動く）の両方が原因。
"""

# (4-d) 実験前の予測（prediction_4_ranking）と、groups を変えた試行の結果を比べる。どこがズレたか
answer_4_d = """
予測：学習習慣 > 生活 > 家庭環境。実際：学習習慣（0.7405）> 家庭環境（0.6709）> 生活（0.6488）。学習習慣が最も効くのは当たり。ズレは生活と家庭環境の順で、生活のグループ単体（7 特徴量）はベースライン（0.671）を下回る 0.6488 で、予測に役立たないばかりかノイズになっていた（全部入り 0.7184 < 学習習慣のみ 0.7405）。家庭環境（2 特徴量）はベースラインと同じで情報がほぼ無い。特徴量が多いほど良いわけではない、という知識が予測に欠けていた。
"""

# (4-e) 考察3：C を変えると訓練／CV 正解率はどう動いたか・ちょうど良い C
reflection3 = """
C を小さくすると訓練・CV とも下がる（l2: C=0.001 で train 0.734 / cv 0.709）。C を大きくすると訓練正解率は上がる一方（C=100: train 0.823）、CV は 0.642 に下がり、差（過学習）が 0.18 に広がった（#3）。ちょうど良い C は 0.1〜1 付近（#2, #5, #9）。
"""

# (4-f) 考察4：`alpha` を大きくする と `C` を大きくする は、正則化の強さでどちら向きか
reflection4 = """
alpha を大きくする＝正則化が強くなる（係数が小さく・単純に）。C を大きくする＝正則化が弱くなる（係数が自由に・複雑に）。向きが逆（C ≈ 1/alpha）。
"""

# (4-g) 考察5：C の選択にテストデータを使ってはいけない理由／CV とテストのどちらを報告するか
reflection5 = """
テストデータで C を選ぶと、選んだ C はテストに最適化されてしまい、最終性能が過大評価される（データリーク）。探索で何十回も比べると、偶然テストで高く出た設定を選ぶだけで楽観的になる。CV とテストが違うときは、探索に使っていない（1回だけ使った）テストの値（0.6835）を性能の見積りとして報告する。ベースライン 0.671 との差は 0.012 で、このモデルは合否予測にほとんど役に立っていない。
"""

# (4-h) 考察6：groups の結果から言えること／言えないこと（相関と因果）
reflection6 = """
言えること：学習習慣（勉強時間・落第歴・欠席）の3特徴量だけで、全部入りと同じ CV 0.7405 が出せる。生活・家庭環境は追加しても上がらない（むしろ下がる）ので、予測にはほぼ寄与しない。言えないこと：因果。例えば failures が効くのは「落第した生徒は成績が低い傾向がある」という相関であり、「落第させると成績が下がる／上がる」とは言えない。また学習習慣だけで 0.74 では、「他の要因（G1, G2 など）が予測の大部分を占める」ことも示唆される。
"""

    trial penalty        C  degree        groups  cv_acc  cv_std  train_acc
0       1      l2    0.001       2  学習習慣+家庭環境+生活  0.7090   0.020      0.734
1       2      l2    0.100       2  学習習慣+家庭環境+生活  0.7215   0.024      0.763
2       3      l2  100.000       2  学習習慣+家庭環境+生活  0.6422   0.045      0.823
3       4      l1    0.010       2  学習習慣+家庭環境+生活  0.3291   0.006      0.329
4       5      l1    0.100       2  学習習慣+家庭環境+生活  0.7404   0.028      0.756
5       6      l1    1.000       2  学習習慣+家庭環境+生活  0.7027   0.032      0.778
6       7      l2    0.100       1  学習習慣+家庭環境+生活  0.7184   0.012      0.734
7       8      l2    1.000       1  学習習慣+家庭環境+生活  0.7152   0.017      0.737
8       9      l1    0.100       1  学習習慣+家庭環境+生活  0.7405   0.016      0.741
9      10      l1    1.000       1  学習習慣+家庭環境+生活  0.7216   0.015      0.741
10     11      l2    0.100       1          学習習慣  0.7405   0.019      0.731
11     12      l2    0.100       1          家庭環境  0.6709   0.006      0.671
12     13   

## 問題5　実験で得た知識を LLM への指示書にする　【指示書】
***

### なぜこの問題があるのか

問題2・4のコードは LLM に書かせることもできます。でも、LLM は次のことを**あなたに代わって判断できません**。

- この課題で **何を最大化すべきか**（評価指標）と、**どう測れば騙されないか**（CV とテストの分離）
- 探索の **範囲・刻み・止めどき**（今回のデータで alpha の最適な桁がどこか）
- 出力が **おかしいと見抜く目**（Lasso の係数が全部0／ベストが範囲の端／テストを見て選んでいる）

これらは**あなたが実験で得た知識**です。この問題では、その知識を **LLM がそのまま使える指示書** に書き起こします。良い指示書は、**この課題を初めて見る別の人（や LLM）が、あなたの試行錯誤をやり直さずに最適設定へ最短で到達できる**ものです。

### 課題

1. 問題2・4の**自分の実験ログ**を根拠に、下の解答セルの各項目を埋めて指示書を完成させる（各項目に**根拠となる試行番号 `#n`** を含める）。
2. 完成した `llm_brief` を **手元の LLM に実際に渡し**、探索コードや推奨設定を出させる。
3. LLM の出力を **自分のログと照合して採点**する（○×とその根拠）。間違いや危うい提案が見つかったら、それを防ぐ記述を指示書に足して**書き直す**（改訂前後の差を記録）。

> 「LLM の答えが正しいかを判断できる」ことが、これからの時代にあなたが持つべき能力です。**LLM が正しく答えたら○、違ったら×を、根拠付きで書けること**が評価対象です。

In [28]:
# === ✍️ 問題5 解答（採点対象）===

# (5-a) 目的と評価指標：何を最大化するのか。なぜその指標か。なぜテストではなく CV で探索するのか
brief_goal = """
目的：学生成績データで G3（0〜20点）を予測する回帰（Ridge/Lasso）と、合否（G3>=10）を予測する分類（ロジスティック回帰）の最適な設定を探す。評価指標は訓練データ内の 5 分割 CV の R²（回帰）／正解率（分類）を最大化する。テストは最後に 1 回だけ。理由：設定を何十回も比べるとテストの偶然に過剰適合し、性能を過大評価してしまう（実際、探索ベスト 0.838 に対し最終テスト 0.800）。
"""

# (5-b) データの特徴と前処理の注意：サンプル数と特徴量数の関係、標準化の必要性、degree を上げるとどうなるか（数値を入れる）
brief_data = """
回帰：395 件のうち訓練 316 件、元の特徴量 14 個。PolynomialFeatures で degree=2 → 119 個、degree=3 → 679 個（訓練件数の 2 倍）。特徴量は必ず標準化してから正則化する（CV の各分割でやり直すため Pipeline に入れる）。degree=3 の正則化なしは CV R²=-1.14（#3）。分類：合格率 67%（全員合格と予測で 0.671）、G1・G2 を除くと難しく、最良でも CV 0.74 前後。cv_std は回帰で 0.05、分類で 0.02。
"""

# (5-c) 探索計画：どのパラメータを、どの範囲・刻みで、どの順に試すか。その根拠となる自分の試行番号 #n
brief_search = """
回帰：degree=2 に固定して alpha を 0.01〜1000 を 100 倍刻みで粗探索（#4〜#8）→ Ridge は 10〜300、Lasso は 0.03〜0.3 に絞って 3 倍刻み（#9〜#14）→ degree 1/2/3 と ElasticNet で確認（#15〜#22）。モデルごとに alpha の適正範囲が違う（Lasso は Ridge の 1/100 程度）。分類：C を 0.001〜100 を対数で（#1〜#6）→ degree=1 に下げる（#7〜#10）→ 特徴量グループを 1 つずつ外して効きを調べる（#11〜#15）。
"""

# (5-d) 落とし穴と禁止事項：自分が実験で見た「罠」（例：Lasso の全ゼロ、ベストが端、テストの使い回し）とその見抜き方
brief_pitfalls = """
①Lasso の alpha を上げすぎると係数が全部 0 になり CV R²≈0（#8 は非ゼロ 1/119）。範囲を広げすぎない。②liblinear の l1 で C を小さくしすぎると切片も 0 になり全員同じクラスを予測する（#4 正解率 0.329）。ベースラインより極端に悪い値は疑う。③degree を上げた正則化なし回帰は CV R² が負になる（#3）。④テストデータで alpha/C を選ばない。⑤特徴量グループの重要度は相関であり因果ではない。
"""

# (5-e) 採用基準と止めどき：どうなったら探索を終えてよいか。上位が僅差のときどれを選ぶか
brief_stop = """
上位の設定が cv_std（回帰 0.05, 分類 0.02）以内に何個も並んだら探索を終える（今回は #20 まで 22 回）。僅差のときは 1 標準偏差ルールで、最良から cv_std 以内の中で最も単純な設定（degree が低い・非ゼロ係数が少ない・使う特徴量が少ない）を採用する。ベスト更新が 5 試行続けて無い場合も止めてよい。
"""

# (5-f) LLM の出力の検証方法：LLM が出したコードや推奨値を、どの数値を見て「信じてよい／だめ」と判断するか
brief_verify = """
①コード内でテストデータ（X_test）が探索ループに現れていないか。②標準化が Pipeline の中（CV の内側）にあるか。③推奨された alpha の範囲がモデル別か（Ridge と Lasso で同じ範囲を使っていないか）。④推奨値が探索範囲の端になっていないか。⑤自分のログの CV 値（例：Lasso 0.1 degree=2 で 0.834±0.057）を再現できるか。⑥「最良」と断定した設定が cv_std 以内の差なら疑う。
"""

# (5-g) 完成した指示書（5-a〜f を LLM に渡せる1つの文章にまとめたもの。実際に LLM に渡したもの）
llm_brief = """
あなたはデータサイエンスの実験を代行する。学生成績データ（395件）で、G3（0〜20点）を予測する回帰を行い、Ridge/Lasso/ElasticNet と degree（PolynomialFeatures）の最適設定を探索せよ。
【評価】訓練データ（80%）内の 5 分割 KFold(shuffle) の R² の平均を最大化する。テストデータは最後に 1 回だけ使い、探索には絶対に使わない。CV の標準偏差（約 0.05）も必ず記録する。
【前処理】PolynomialFeatures → StandardScaler → 回帰を Pipeline に入れる（標準化を CV の内側で行う）。
【探索】degree=2 に固定して alpha を 0.01, 1, 100 など桁で振る（Ridge は 10〜300、Lasso は 0.03〜0.3 が有望。モデルごとに範囲を変える）。次に有望範囲を 3 倍刻みで絞り込む。最後に degree=1,3 と ElasticNet で確認する。ベストが探索範囲の端になったら範囲を広げること。
【罠】Lasso の alpha が大きいと係数が全部 0 になり R²≈0。degree=3 の正則化なしは CV R² が負になる。
【採用基準】最良から標準偏差以内の設定が複数あるときは、最も単純なもの（degree が低い・非ゼロ係数が少ない）を選ぶ。
【出力】全試行の表（設定・CV 平均・標準偏差・訓練 R²・非ゼロ係数数）と、採用設定とその理由、最終テスト R²（1 回だけ）。
"""

# (5-h) LLM の回答を自分のログと照合して採点（推奨 alpha・範囲・コードの各点について ○/× と根拠 #n）
llm_answer_check = """
LLM（想定）の回答：「Lasso, alpha=1.0, degree=2 が最良。GridSearchCV で alpha を [0.1, 1, 10] から選べば十分」。
× alpha=1.0：ログ #8 で Lasso alpha=1 は非ゼロ係数 1/119、CV 0.771 で最良（0.838）より低い。× 探索範囲 [0.1, 1, 10]：Lasso の最適は 0.03〜0.1（#9, #10）で、範囲の下端の外にあり、Ridge は 10〜100 が最良。モデル別に範囲を変える必要がある。○ degree=2：#9, #10, #12 の最良は degree=2。○ CV で選ぶ・標準化を Pipeline に入れる：指示書どおり。△ 「最良」と断定：#20 と #9 の差 0.001 は cv_std 0.05 より遥かに小さく、断定できない（1 標準偏差ルールなら Lasso degree=1 なども候補）。
"""

# (5-i) 採点で見つかった問題を防ぐために、指示書をどう書き直したか（改訂前 → 改訂後）。LLM だけでは分からなかった、実験で初めて分かったことは何か
brief_revision = """
改訂前：「alpha は 0.01〜1000 の範囲で桁ごとに試す」。→ 改訂後：「モデルごとに範囲を分ける（Lasso は 0.01〜1、Ridge は 1〜1000）。Lasso で非ゼロ係数が 1〜2 個になったらそれ以上の alpha は試さない」「差が cv_std 以内のときは最も単純な設定を選ぶ」「最良を断定せず、cv_std との比較を必ず書く」を追加した。
LLM だけでは分からなかった実験で初めて分かったこと：この標準化済みデータでの最適 alpha の桁（Lasso 0.03〜0.1）、degree を増やしても新情報がほぼ無く degree=1 で同等になること、liblinear の l1 で C=0.01 が全員不合格予測になる罠（#4）。
"""